# TRAIN LẠI TOÀN BỘ MODEL MŨ BẢO HIỂM v2 - ĐA GÓC QUAY CẦU VƯỢT
Dataset v2 có:
- **160 ảnh đa góc rẽ** (xe đi thẳng + rẽ trái + rẽ phải + sát lề + chở 2 người) x 3 bản sao = 420 mẫu bridge thực tế
- **65 ảnh cầu vượt lần trước** x 2 = 130 mẫu
- **250 ảnh Roboflow** (giữ kiến thức mũ màu sắc đa dạng)
- **Tổng Train: 800 ảnh | Val: 60 ảnh**

---
**BƯỚC THỰC HIỆN:**
1. Bật GPU T4: **Notebook options → Accelerator → GPU T4**
2. Upload file `dataset_v2_train.zip` (18.2 MB) và `yolo11s_helmet_lp.pt` vào Input
3. Bấm **Run All**

In [ ]:
!nvidia-smi
!pip install -q ultralytics

In [ ]:
import os, glob, shutil, zipfile

work_dir = '/kaggle/working'
dest = os.path.join(work_dir, 'dataset_v2_train')
os.makedirs(dest, exist_ok=True)

# Tim file zip dataset v2
zips = glob.glob('/kaggle/input/**/dataset_v2_train*.zip', recursive=True)
if zips:
    print(f'Tim thay zip: {zips[0]}')
    with zipfile.ZipFile(zips[0], 'r') as zf:
        zf.extractall(work_dir)
else:
    # Thu tim thu muc da giai nen
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'data.yaml' in files:
            shutil.copytree(root, dest, dirs_exist_ok=True)
            break

# Dinh vi data.yaml va sua path tuyet doi
yaml_path = None
for root, dirs, files in os.walk(work_dir):
    if 'data.yaml' in files:
        yaml_path = os.path.join(root, 'data.yaml')
        folder = os.path.dirname(os.path.abspath(yaml_path))
        with open(yaml_path, 'r', encoding='utf-8') as f:
            content = f.read()
        # Sua tat ca cac dong path
        lines = []
        for line in content.split('\n'):
            if line.strip().startswith('path:'):
                lines.append(f"path: '{folder}'")
            else:
                lines.append(line)
        with open(yaml_path, 'w', encoding='utf-8') as f:
            f.write('\n'.join(lines))
        break

print('data.yaml san sang:', yaml_path)
assert yaml_path and os.path.exists(yaml_path)

In [ ]:
from ultralytics import YOLO
import glob

# Nap model cu de hoc tiep (Transfer Learning nhanh hon 3 lan so voi train tu dau)
base_pts = glob.glob('/kaggle/input/**/yolo11s_helmet_lp.pt', recursive=True)
base_model = base_pts[0] if base_pts else 'yolo11s.pt'
print(f'Base model: {base_model}')

model = YOLO(base_model)

results = model.train(
    data=yaml_path,
    epochs=40,
    imgsz=640,
    batch=16,
    patience=10,
    lr0=0.001,
    lrf=0.01,
    # Augmentation dac thu goc cau vuot
    degrees=15.0,        # Xoay +/- 15 do muc mo re trai/re phai
    perspective=0.0005,  # Do nghieng phoi canh khi xe re goc
    hsv_v=0.4,           # Thay doi do sang chong nang choi / bong ram
    hsv_s=0.7,           # Thay doi do bao hoa mau
    flipud=0.1,          # Lật dọc nhe de tang bien the
    mosaic=0.5,          # Mosaic training giup nhan dien xe nho
    save=True,
    project='runs_v2',
    name='helmet_multiangle',
    device=0
)
print('HOAN TAT TRAIN v2!')

In [ ]:
import shutil, glob, os

pts = glob.glob('/kaggle/working/**/best.pt', recursive=True)
print('Tim thay best.pt:', pts)

if pts:
    shutil.copy2(pts[-1], '/kaggle/working/yolo11s_helmet_lp.pt')
    sz = os.path.getsize('/kaggle/working/yolo11s_helmet_lp.pt') / (1024*1024)
    print(f'DA XUAT THANH CONG yolo11s_helmet_lp.pt ({sz:.1f} MB)')
    print('=> Refresh Output o cot ben phai va Download file nay!')